In [7]:
import pandas as pd
import numpy as np

fact = pd.read_csv('data/processed/fact_harga_pangan.csv')
dim_negara = pd.read_csv('data/processed/dim_negara.csv')
dim_komoditas = pd.read_csv('data/processed/dim_komoditas.csv')
dim_waktu = pd.read_csv('data/processed/dim_waktu.csv')

print("1. SPARSITY")
total_cells = fact.shape[0] * fact.shape[1]
null_cells = fact.isnull().sum().sum()
sparsity = (null_cells / total_cells) * 100
print(f"Total sel       : {total_cells:,}")
print(f"Sel kosong (NaN): {null_cells:,}")
print(f"Sparsity        : {sparsity:.2f}%")
print()
print("Null per kolom:")
print(fact.isnull().sum())

print()
print("2. KELENGKAPAN KOMBINASI")
n_waktu = len(dim_waktu)
n_negara = len(dim_negara)
n_komoditas = len(dim_komoditas)
theoretical_max = n_waktu * n_negara * n_komoditas
actual_rows = len(fact)
coverage = (actual_rows / theoretical_max) * 100
print(f"Kombinasi teoritis (waktu×negara×komoditas): {theoretical_max:,}")
print(f"Baris aktual di fact table                 : {actual_rows:,}")
print(f"Coverage                                   : {coverage:.2f}%")
print(f"Sparsity kombinasi                         : {100 - coverage:.2f}%")

print()
print("3. Distribusi per negara")
negara_dist = fact.groupby('negara_id').size().reset_index(name='count')
negara_dist = negara_dist.merge(dim_negara[['negara_id','country_name']], on='negara_id')
negara_dist['pct'] = (negara_dist['count'] / negara_dist['count'].sum() * 100).round(2)
print(negara_dist[['country_name','count','pct']].to_string(index=False))
print(f"\nRatio max/min: {negara_dist['count'].max() / negara_dist['count'].min():.2f}x")

print()
print("4. Distribusi per komoditas")
komoditas_dist = fact.groupby('komoditas_id').size().reset_index(name='count')
komoditas_dist = komoditas_dist.merge(dim_komoditas[['komoditas_id','commodity_name']], on='komoditas_id')
komoditas_dist = komoditas_dist.sort_values('count', ascending=False)
print("Top 10 komoditas terbanyak:")
print(komoditas_dist.head(10)[['commodity_name','count']].to_string(index=False))
print("\nBottom 10 komoditas tersedikit:")
print(komoditas_dist.tail(10)[['commodity_name','count']].to_string(index=False))

print()
print("5. OUTLIER avg_price")
q1 = fact['avg_price'].quantile(0.25)
q3 = fact['avg_price'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
outliers = fact[(fact['avg_price'] < lower) | (fact['avg_price'] > upper)]
print(f"Q1              : {q1:.4f}")
print(f"Q3              : {q3:.4f}")
print(f"IQR             : {iqr:.4f}")
print(f"Batas bawah     : {lower:.4f}")
print(f"Batas atas      : {upper:.4f}")
print(f"Jumlah outlier  : {len(outliers):,} dari {len(fact):,} baris ({len(outliers)/len(fact)*100:.2f}%)")

print()
print("6. DISTRIBUSI avg_price")
print(fact['avg_price'].describe().round(4))
print(f"Skewness : {fact['avg_price'].skew():.4f}")
print(f"Kurtosis : {fact['avg_price'].kurtosis():.4f}")

print()
print("7. DISTRIBUSI TEMPORAL")
waktu_dist = fact.groupby('waktu_id').size().reset_index(name='count')
waktu_dist = waktu_dist.merge(dim_waktu[['waktu_id','periode']], on='waktu_id')
print(waktu_dist[['periode','count']].to_string(index=False))

1. SPARSITY
Total sel       : 46,008
Sel kosong (NaN): 7,838
Sparsity        : 17.04%

Null per kolom:
waktu_id                0
negara_id               0
komoditas_id            0
avg_price               0
min_price               0
max_price               0
record_count            0
fp_cpi_totl_zg       1903
ny_gdp_mktp_kd_zg    1819
ny_gdp_pcap_cd       1819
tm_val_food_zs_un    2297
tahun_partisi           0
dtype: int64

2. KELENGKAPAN KOMBINASI
Kombinasi teoritis (waktu×negara×komoditas): 21,888
Baris aktual di fact table                 : 3,834
Coverage                                   : 17.52%
Sparsity kombinasi                         : 82.48%

3. Distribusi per negara
country_name  count   pct
    Cambodia   1018 26.55
   Indonesia    282  7.36
     Lao PDR    528 13.77
     Myanmar    161  4.20
 Philippines   1416 36.93
 Timor-Leste    429 11.19

Ratio max/min: 8.80x

4. Distribusi per komoditas
Top 10 komoditas terbanyak:
            commodity_name  count
                  

In [8]:
import json
import glob

# Cek nama negara di World Bank JSON
files = glob.glob('data/raw/raw_worldbank_batch_*.json')
for f in files[:1]:
    with open(f) as fp:
        data = json.load(fp)
    countries = list(set([r['country']['value'] for r in data if r.get('value')]))
    print("Nama negara di World Bank:")
    for c in sorted(countries):
        print(f"  '{c}'")

# Cek nama negara di dim_negara
import pandas as pd
dim_negara = pd.read_csv('data/processed/dim_negara.csv')
print("\nNama negara di dim_negara:")
print(dim_negara['country_name'].tolist())

Nama negara di World Bank:
  'Cambodia'
  'Indonesia'
  'Lao PDR'
  'Malaysia'
  'Philippines'
  'Singapore'
  'Thailand'
  'Timor-Leste'
  'Viet Nam'

Nama negara di dim_negara:
['Cambodia', 'Indonesia', 'Lao PDR', 'Myanmar', 'Philippines', 'Timor-Leste']


In [9]:
fact = pd.read_csv('data/processed/fact_harga_pangan.csv')
print(fact.columns.tolist())
print(fact['tahun_partisi'].isna().sum(), "baris NULL dari", len(fact))

['waktu_id', 'negara_id', 'komoditas_id', 'avg_price', 'min_price', 'max_price', 'record_count', 'fp_cpi_totl_zg', 'ny_gdp_mktp_kd_zg', 'ny_gdp_pcap_cd', 'tm_val_food_zs_un', 'tahun_partisi']
0 baris NULL dari 3834
